In [2]:
import time
import boto3
import pandas as pd

In [3]:
athena = boto3.client(
    "athena",
    endpoint_url="http://localhost:4566",
    region_name="us-east-1",
    aws_access_key_id="test",
    aws_secret_access_key="test",
)

In [4]:
athena.list_query_executions(MaxResults=1)

{'QueryExecutionIds': ['aa00ee82-44ca-456e-855e-b1344fb01244',
  'ea1bde48-323c-4542-9214-6f4a73dfe3f5'],
 'ResponseMetadata': {'RequestId': 'c1ef004d-3978-4788-91ad-20bfe21f652f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.1;charset=UTF-8',
   'content-length': '101',
   'date': 'Wed, 04 Nov 2026 01:45:57 GMT',
   'x-amz-id-2': 'c1ef004d-3978-4788-91ad-20bfe21f652f',
   'x-amz-request-id': 'c1ef004d-3978-4788-91ad-20bfe21f652f',
   'x-amzn-requestid': 'c1ef004d-3978-4788-91ad-20bfe21f652f'},
  'RetryAttempts': 0}}

In [7]:
def query(sql):
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": "research"},
        ResultConfiguration={
            "OutputLocation": "s3://data-platform-athena-results/queries/"
        },
    )
    query_id = response["QueryExecutionId"]

    while True:
        execution = athena.get_query_execution(QueryExecutionId=query_id)
        status = execution["QueryExecution"]["Status"]
        if status["State"] == "SUCCEEDED":
            break
        if status["State"] in ("FAILED", "CANCELLED"):
            raise RuntimeError(status.get("StateChangeReason") or status["State"])
        time.sleep(0.5)

    pages = athena.get_paginator("get_query_results").paginate(
        QueryExecutionId=query_id
    )
    rows = (
        [cell.get("VarCharValue") for cell in row["Data"]]
        for page in pages
        for row in page["ResultSet"]["Rows"]
    )
    columns = next(rows)  # The first result row contains the column names.
    return pd.DataFrame(rows, columns=columns)

In [10]:
query("SELECT author_type, count(*) as c from prs group by author_type order by author_type")

,author_type,c
0,agent,1
1,human,1


In [11]:
# after wer added the second csv file into the s3 bucket
query("SELECT author_type, count(*) as c from prs group by author_type order by author_type")

,author_type,c
0,agent,3
1,human,2
